# EmotionVision AI — CNN Training (FER-2013)

Train the CNN once in Google Colab, save the model and labels, then use them in the Streamlit webcam app.

**Dataset expected:** a ZIP containing `train/<emotion>/images` and `test/<emotion>/images` (the uploaded archive follows this structure).

**Runtime:** Colab → Runtime → Change runtime type → GPU (recommended, not mandatory).

This model predicts facial-expression categories, not a person's true internal sentiment.

In [ ]:
# 1. Imports and configuration
import os
import json
import zipfile
from pathlib import Path

import tensorflow as tf
import matplotlib.pyplot as plt

IMG_SIZE = 48
BATCH_SIZE = 64
EPOCHS = 25

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

BASE_DIR = Path("/content/emotionvision")
DATA_DIR = BASE_DIR / "dataset"
MODEL_DIR = BASE_DIR / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 2. Upload and extract the dataset ZIP
from google.colab import files

uploaded = files.upload()  # Select the provided archive ZIP
zip_name = next(iter(uploaded.keys()))

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Dataset extracted to:", DATA_DIR)
print("Train folder exists:", (DATA_DIR / "train").exists())
print("Test folder exists:", (DATA_DIR / "test").exists())

if not (DATA_DIR / "train").exists() or not (DATA_DIR / "test").exists():
    raise FileNotFoundError(
        "Expected train/ and test/ folders inside the ZIP. "
        "Check the ZIP's internal folder structure."
    )


In [ ]:
# 3. Load training, validation, and test images
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=42,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=42,
    label_mode="int"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="int"
)

class_names = train_ds.class_names
print("Emotion labels:", class_names)

with open(MODEL_DIR / "labels.json", "w") as file:
    json.dump(class_names, file)


In [ ]:
# 4. Normalize images and improve loading performance
normalization = tf.keras.layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(lambda images, labels: (normalization(images), labels))
val_ds = val_ds.map(lambda images, labels: (normalization(images), labels))
test_ds = test_ds.map(lambda images, labels: (normalization(images), labels))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


In [ ]:
# 5. Build the CNN model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(48, 48, 1)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# 6. Train the model
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_emotion_model.keras"),
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


In [ ]:
# 7. Evaluate on the separate test dataset
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")


In [ ]:
# 8. Save the trained model and label names
# Use the best validation checkpoint for final evaluation and deployment.
best_model_path = MODEL_DIR / "best_emotion_model.keras"
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)

final_model_path = MODEL_DIR / "emotion_model.keras"
model.save(final_model_path)

print("Saved model:", final_model_path)
print("Saved labels:", MODEL_DIR / "labels.json")


In [ ]:
# 9. Plot and save accuracy
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(MODEL_DIR / "accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot and save loss
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig(MODEL_DIR / "loss.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 10. Package the files needed by the Streamlit app and download
from google.colab import files

export_zip = "/content/emotionvision_model_files.zip"

with zipfile.ZipFile(export_zip, "w", zipfile.ZIP_DEFLATED) as zip_ref:
    zip_ref.write(MODEL_DIR / "emotion_model.keras", "models/emotion_model.keras")
    zip_ref.write(MODEL_DIR / "labels.json", "models/labels.json")
    zip_ref.write(MODEL_DIR / "accuracy.png", "models/accuracy.png")
    zip_ref.write(MODEL_DIR / "loss.png", "models/loss.png")

files.download(export_zip)
